# 3.1 · Tools: capabilities the model can invoke

**工具（Tools）：模型可调用的能力**

> **After this lesson you can:** Define MCP tools with precise schemas, return text, media, links and structured content, report errors the model can recover from, and keep a host's tool list fresh.

Official reading: [Understanding MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/learn/server-concepts) · [Spec: Tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A server with one tool per result type
`MCPServer` derives everything from the function: name, docstring → `description`, type hints → `inputSchema`, and (for typed returns) `outputSchema` + `structuredContent`. `title` and `annotations` are decorator arguments. We silence the SDK's server-side logging so the error demos below stay readable.

In [ ]:
import base64
import json
import logging
import struct
import zlib

from pydantic import BaseModel, Field

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context, Image
from mcp.server.mcpserver.exceptions import ToolError

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def show(model, limit=1500):
    """Print a pydantic protocol object exactly as it goes on the wire (minus serverInfo noise)."""
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.get("_meta", {}).pop("io.modelcontextprotocol/serverInfo", None)
    if not data.get("_meta"):
        data.pop("_meta", None)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:limit])


def tiny_png() -> bytes:
    """A valid 1x1 red PNG, built by hand so the notebook needs no image library."""
    chunk = lambda kind, body: (struct.pack(">I", len(body)) + kind + body
                                + struct.pack(">I", zlib.crc32(kind + body) & 0xFFFFFFFF))
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", 1, 1, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(b"\x00\xff\x00\x00")) + chunk(b"IEND", b""))


class Weather(BaseModel):
    temperature: float = Field(description="Temperature in celsius")
    conditions: str
    humidity: float = Field(description="Humidity percentage")


server = MCPServer("travel")


@server.tool()
def greet(name: str) -> str:
    """Say hello. Returns plain text."""
    return f"Hello, {name}!"


@server.tool(title="Weather Data Retriever",
             annotations=types.ToolAnnotations(readOnlyHint=True, openWorldHint=True))
def get_weather_data(location: str) -> Weather:
    """Get current weather data for a location (city name or zip code)."""
    return Weather(temperature=22.5, conditions="Partly cloudy", humidity=65)


@server.tool(annotations=types.ToolAnnotations(readOnlyHint=True))
def render_chart() -> list[types.ContentBlock]:
    """Render the forecast chart as an image meant for the user, plus a caption for the model."""
    return [
        types.ImageContent(type="image", data=base64.b64encode(tiny_png()).decode(), mime_type="image/png",
                           annotations=types.Annotations(audience=["user"], priority=0.9)),
        types.TextContent(type="text", text="Chart rendered: 7-day forecast, mostly sunny."),
    ]


@server.tool()
def find_itinerary(trip: str) -> list[types.ContentBlock]:
    """Return a link to the itinerary file and the packing list inline."""
    return [
        types.ResourceLink(type="resource_link", uri=f"trips://{trip}/itinerary.md", name="itinerary.md",
                           description="Full day-by-day plan", mime_type="text/markdown"),
        types.EmbeddedResource(type="resource", resource=types.TextResourceContents(
            uri=f"trips://{trip}/packing.txt", mime_type="text/plain", text="passport\nadapter\nsunscreen")),
    ]


@server.tool(annotations=types.ToolAnnotations(readOnlyHint=False, destructiveHint=True, idempotentHint=False))
def cancel_booking(booking_id: str) -> str:
    """Cancel a booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


print("tools registered:", [t.name for t in await server.list_tools()])

### 2. Discovery: what the model will see
`tools/list` returns the definitions in registration order (deterministic). Look at the generated `inputSchema`, the `outputSchema` derived from the pydantic model, and the annotations. Also note `ttlMs` / `cacheScope` on the result (lesson 3.4).

In [ ]:
async with Client(server) as client:
    listing = await client.list_tools()
    print("capabilities.tools:", client.server_capabilities.tools)
    print("order:", [t.name for t in listing.tools], "| ttlMs:", listing.ttl_ms, "| cacheScope:", listing.cache_scope)
    show(next(t for t in listing.tools if t.name == "get_weather_data"), limit=2500)

Display-name precedence is `title` → `annotations.title` → `name`. A host would label the buttons like this:

In [ ]:
def display_name(tool: types.Tool) -> str:
    return tool.title or (tool.annotations.title if tool.annotations else None) or tool.name


async with Client(server) as client:
    for t in (await client.list_tools()).tools:
        a = t.annotations
        hints = a.model_dump(by_alias=True, exclude_none=True) if a else {}
        print(f"{display_name(t):<24} hints={hints}")

### 3. Call every tool and read the content blocks
Plain text, structured content (with its text mirror), an image annotated for the user, a resource link and an embedded resource.

In [ ]:
async with Client(server) as client:
    for name, args in [("greet", {"name": "Ada"}), ("get_weather_data", {"location": "Barcelona"}),
                       ("render_chart", {}), ("find_itinerary", {"trip": "bcn-2026"})]:
        result = await client.call_tool(name, args)
        print(f"--- tools/call {name}")
        show(result, limit=900)

For `get_weather_data` the SDK filled **both** `structuredContent` (for programs) and a text block with the same JSON (for the model and older clients) — exactly the spec's SHOULD. For `greet`, a `str` return is wrapped as `{"result": ...}` because a function return type always yields an object-shaped schema in this SDK.

### 4. Break it: a server that lies about its output
Clients SHOULD validate `structuredContent` against `outputSchema`. The SDK's client does this inside `call_tool`. Here we hand its validator a forged result that doesn't match the schema, to see what a buggy or malicious server would trigger.

In [ ]:
async with Client(server) as client:
    await client.list_tools()  # the client remembers each tool's outputSchema
    forged = types.CallToolResult(content=[types.TextContent(type="text", text="it's warm")],
                                  structured_content={"temperature": "warm", "conditions": "sunny", "humidity": 65})
    try:
        await client.session.validate_tool_result("get_weather_data", forged)
    except RuntimeError as exc:
        print("client rejected the result:\n", str(exc).splitlines()[0])

### 5. Two kinds of errors
Four failure modes. Watch which become `isError` results (visible to the model) and how much each message tells the model.

In [ ]:
@server.tool()
def book_flight(origin: str, destination: str, date: str) -> str:
    """Book a flight. date must be YYYY-MM-DD and in the future."""
    if not date[:4].isdigit() or date < "2026-07-28":
        raise ToolError(f"Invalid departure date {date!r}: must be YYYY-MM-DD and in the future.")
    return f"Booked {origin}->{destination} on {date}"


@server.tool()
def flaky_api() -> str:
    """Calls an upstream API that crashes."""
    raise ConnectionError("upstream 10.0.0.7:5432 refused connection (password=hunter2)")


async with Client(server) as client:
    cases = [("business error (ToolError)", "book_flight", {"origin": "NYC", "destination": "BCN", "date": "08/08/2025"}),
             ("unexpected exception", "flaky_api", {}),
             ("arguments fail inputSchema", "book_flight", {"origin": "NYC"}),
             ("unknown tool", "teleport", {})]
    for label, name, args in cases:
        r = await client.call_tool(name, args)
        print(f"{label:<28} isError={r.is_error} -> {r.content[0].text.splitlines()[0][:110]}")

Note two SDK behaviours. A `ToolError` message reaches the model (actionable), while an unexpected exception becomes a generic "Error executing tool flaky_api" — internal details such as hosts and passwords are **not** leaked, which is what "sanitize tool outputs" asks for. And this SDK reports an unknown tool as `isError` too, whereas the spec lists it as a protocol error.

### 6. Protocol errors on the wire
To see a real JSON-RPC error we go below the SDK: run the server as a stdio subprocess and send a `tools/call` that violates the `CallToolRequest` schema (no `name`).

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "weather_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("weather")

    @server.tool()
    def get_weather(location: str) -> str:
        """Get current weather information for a location."""
        return f"Current weather in {location}: 22C, partly cloudy"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
proc = subprocess.Popen([sys.executable, str(workdir / "weather_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(id_, method, params):
    proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": id_, "method": method, "params": params}) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    reply.get("result", {}).pop("_meta", None)   # hide serverInfo noise
    print(f">>> {method} {json.dumps({k: v for k, v in params.items() if k != '_meta'})}")
    print("<<<", json.dumps(reply), "\n")


rpc(1, "tools/call", {"name": "get_weather", "arguments": {"location": "New York"}, "_meta": META})
rpc(2, "tools/call", {"arguments": {"location": "New York"}, "_meta": META})   # malformed: no name
proc.stdin.close()
_ = proc.wait(timeout=10)

### 7. Tool names
The SDK ships the spec's naming check (it only *warns* at registration). Try a few names.

In [ ]:
from mcp.shared.tool_name_validation import validate_tool_name

for name in ["getUser", "DATA_EXPORT_v2", "admin.tools.list", "search flights", "créer_événement", "x" * 129]:
    r = validate_tool_name(name)
    print(f"{name[:30]!r:<34} valid={r.is_valid!s:<5} {r.warnings[:1]}")

### 8. Stateful tools with explicit handles
No sessions, so state lives under an opaque handle that the model carries between calls. An unknown handle is an `isError` result that tells the model how to recover.

In [ ]:
import uuid

baskets: dict[str, list[str]] = {}
shop = MCPServer("shop")


class BasketRef(BaseModel):
    basket_id: str


@shop.tool()
def create_basket() -> BasketRef:
    """Create an empty basket and return its id. Baskets expire after 24 hours of inactivity."""
    basket_id = f"bsk_{uuid.uuid4().hex[:12]}"
    baskets[basket_id] = []
    return BasketRef(basket_id=basket_id)


@shop.tool()
def add_item(basket_id: str, sku: str) -> list[str]:
    """Add a product to an existing basket; returns the basket contents."""
    if basket_id not in baskets:
        raise ToolError(f"Unknown or expired basket {basket_id!r}. Call create_basket to start a new one.")
    baskets[basket_id].append(sku)
    return baskets[basket_id]


async with Client(shop) as client:
    handle = (await client.call_tool("create_basket", {})).structured_content["basket_id"]
    print("handle:", handle)
    print((await client.call_tool("add_item", {"basket_id": handle, "sku": "TENT-2P"})).structured_content)
    bad = await client.call_tool("add_item", {"basket_id": "bsk_000000000000", "sku": "STOVE"})
    print("isError:", bad.is_error, "->", bad.content[0].text)

### 9. Dynamic tools and `notifications/tools/list_changed`
A tool can change the tool set. In 2026-07-28 the client hears about it only if it opened a `subscriptions/listen` stream with `toolsListChanged` (lesson 3.4 goes deep). `MCPServer` does not notify automatically when you call `add_tool`; the tool publishes the event with `ctx.notify_tools_changed()`.

In [ ]:
@server.tool()
async def enable_hotels(ctx: Context) -> str:
    """Unlock hotel-booking tools for this deployment."""
    def search_hotels(city: str) -> list[str]:
        """Search hotels in a city."""
        return [f"Hotel Arts {city}", f"Casa {city}"]
    server.add_tool(search_hotels)
    await ctx.notify_tools_changed()
    return "hotel tools enabled"


async with Client(server) as client:
    before = [t.name for t in (await client.list_tools()).tools]
    async with client.listen(tools_list_changed=True) as sub:
        print("listening, honored filter:", sub.honored.model_dump(exclude_none=True, by_alias=True))
        await client.call_tool("enable_hotels", {})
        event = await anext(sub)
        print("event:", event)
        after = [t.name for t in (await client.list_tools()).tools]
    print("new tools:", sorted(set(after) - set(before)))

### 10. A host-side approval gate
The protocol leaves UX to the host, but the spec says a human SHOULD be able to deny calls and annotations are untrusted unless the server is. Here is a minimal policy: auto-approve read-only tools **only for trusted servers**, otherwise ask (simulated by a scripted "user").

In [ ]:
TRUSTED_SERVERS = {"travel"}


def needs_approval(server_name: str, tool: types.Tool) -> bool:
    hints = tool.annotations or types.ToolAnnotations()
    if server_name not in TRUSTED_SERVERS:
        return True                      # never trust hints from unknown servers
    return not hints.read_only_hint      # trusted + read-only -> no dialog


def scripted_user(tool_name: str, args: dict) -> bool:
    print(f"   [dialog] Allow {tool_name}({args})?", "-> DENY" if tool_name.startswith("cancel") else "-> allow")
    return not tool_name.startswith("cancel")


async def host_call(client, server_name, name, args):
    tool = next(t for t in (await client.list_tools()).tools if t.name == name)
    if needs_approval(server_name, tool) and not scripted_user(name, args):
        return f"{name}: denied by user (the model is told the call was refused)"
    text = (await client.call_tool(name, args)).content[0].text
    return f"{name}: " + " ".join(text.split())[:70]


async with Client(server) as client:
    for server_name, name, args in [("travel", "get_weather_data", {"location": "Rome"}),
                                    ("travel", "cancel_booking", {"booking_id": "B-42"}),
                                    ("random-server", "get_weather_data", {"location": "Rome"})]:
        print(f"{server_name}/{name}")
        print("  ", await host_call(client, server_name, name, args))

### Your turn
1. Add a tool `convert_currency(amount: float, frm: str, to: str)` whose return type is a pydantic model with `amount` and `rate`. Inspect its `outputSchema`, then make it raise a `ToolError` for an unsupported currency and check that the message reaches the client with `isError: true`.
2. Give `get_weather_data` an explicit `title` in `annotations` *and* the decorator `title`. Which one does `display_name` pick? Remove the decorator title and check again.
3. In section 6, send `tools/call` for a tool name that does not exist over raw stdio. Compare the answer with the spec's `-32602 Unknown tool` example and decide how your host should handle both shapes.
4. Extend section 8 with a `checkout(basket_id)` tool that deletes the basket, then show that a second `add_item` with the same handle produces a recoverable `isError` result.